# 00 · Quickstart

The whole `pymv` pipeline end-to-end in one page: build data, fit the **LMGEC** baseline and the modular **LMGECplus**, and score them. Later notebooks open up each stage.

| Notebook | Stage |
|---|---|
| 00 quickstart | the pipeline at a glance |
| 01 data & preprocessing | loading views, normalize → propagate |
| 02 lmgec baseline | the original algorithm, step by step |
| 03 lmgecplus components | swap losses / attention / clustering |
| 04 ablation study | one-call ablation ladder + tuning |

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")   # quiet TensorFlow

import numpy as np
import scipy.sparse as sp


def make_synthetic(n_per=60, k=3, dims=(30, 20), sep=4.0, seed=0):
    """A small, well-separated multi-view dataset: (Xs, As, labels).

    Each view is Gaussian blobs around k cluster centres, plus a sparse
    intra-cluster adjacency so graph propagation has something to smooth.
    """
    rng = np.random.RandomState(seed)
    n = n_per * k
    labels = np.repeat(np.arange(k), n_per)
    Xs, As = [], []
    for d in dims:
        centres = rng.randn(k, d) * sep
        X = centres[labels] + rng.randn(n, d)
        Xs.append(X.astype(np.float64))
        A = np.zeros((n, n))
        for i in range(n):
            same = np.where(labels == labels[i])[0]
            nbrs = rng.choice(same, size=min(6, len(same)), replace=False)
            A[i, nbrs] = 1.0
        As.append(sp.csr_matrix(np.maximum(A, A.T)))
    return Xs, As, labels


Xs, As, labels = make_synthetic()

# --- To use a real benchmark instead (downloads + caches on first call): ---
# from pymv.data import load_dataset
# data = load_dataset("acm")
# Xs, As, labels = data.Xs, data.As, data.labels

k = len(np.unique(labels))
print(f"views={len(Xs)}  samples={Xs[0].shape[0]}  clusters={k}")
print("view feature dims:", [X.shape[1] for X in Xs])

## Fit both models

`LMGEC` is the original TensorFlow algorithm. `LMGECplus` with no overrides is configured to **reproduce** LMGEC exactly, then lets you replace any component.

In [ ]:
from pymv.models import LMGEC, LMGECplus
from pymv.metrics import clustering_accuracy, nmi_score

lmgec = LMGEC(n_clusters=k, embedding_dim=16, max_iter=30).fit(Xs, As)
plus  = LMGECplus(n_clusters=k, embedding_dim=16, max_iter=30).fit(Xs, As)

for name, m in [('LMGEC', lmgec), ('LMGECplus', plus)]:
    acc = clustering_accuracy(labels, m.labels_)
    nmi = nmi_score(labels, m.labels_)
    print(f'{name:10s}  ACC={acc:.3f}  NMI={nmi:.3f}  alphas={np.round(m.alphas_, 3)}')

## Or run a formatted comparison with `Experiment`

`Experiment` handles repeats, deep-copies each model per run, and prints clean result tables.

In [ ]:
from pymv.experiment import Experiment

Experiment(
    datasets={'synthetic': (Xs, As, labels)},
    models=[
        LMGEC(n_clusters=k, embedding_dim=16, max_iter=30),
        LMGECplus(n_clusters=k, embedding_dim=16, max_iter=30),
    ],
    metrics=[clustering_accuracy, nmi_score],
    runs=3,
).run();